# Mumtaz MDD vs HC — frozen CBraMod (TDBRAIN과 같은 프로토콜)

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → GPU` 선택 후, 위에서부터 한 셀씩 실행하세요.

- **데이터:** Mumtaz et al. (2017) MDD Patients and Healthy Controls EEG Data, NEMAR `nm000114` (BIDS, EDF, 256 Hz, CC-BY-4.0)
- **과제:** 질병 유무 분류 (HC = 0, MDD = 1), 피험자 단위 층화 5-fold (seed 42)
- **선정:** 라벨은 피험자 ID 접두어(HS → 0, MDDS → 1) · eyesOpen(EO)·eyesClosed(EC)만 사용(P300 제외) · 같은 조건 기록이 여러 개면 첫 세션·첫 run · EO/EC가 없는 피험자 제외 → **63명 (HC 30 / MDD 33), 기록 119개**

| 실행 | 내용 |
|---|---|
| EO + EC (주 결과) | 피험자 점수 = EO 평균과 EC 평균의 같은 비중 평균 |
| EO만 / EC만 | 조건별 단독 평가 |
| EO + EC, 앞 120초 | TDBRAIN(약 2분)과 기록 길이를 맞춘 보조 결과 |

**설정:** 19채널(Mumtaz 순서) · 재참조 없음 · 0.3–75 Hz · 50 Hz notch · 200 Hz · 5 s 비중첩 · μV/100 · 100 μV 초과 구간은 표시만 · frozen CBraMod(원본 repo + 원본 사전학습 가중치, 시간 패치 평균 → 3,800차원) · StandardScaler + L2 로지스틱 회귀(C = 0.01), 구간 단위 학습, 임계값 0.5

> **데이터는 저장소에 포함되지 않습니다.**

## 0. 설치

In [ ]:
!pip install -q mne einops huggingface_hub
import torch; print('GPU:', torch.cuda.is_available())

## 1. Google Drive 연결 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob
WORK = '/content/drive/MyDrive/mumtaz_hcmdd_project'   # 결과·데이터 백업 위치
BIDS_ROOT = f'{WORK}/nm000114'                          # BIDS 데이터 위치 (sub-*/.../*_eeg.edf)
OUT = f'{WORK}/out'
os.makedirs(OUT, exist_ok=True)

## 2. 데이터 준비
NEMAR `nm000114`(https://nemar.org)에서 BIDS 데이터를 받아 Drive의 `BIDS_ROOT` 위치에 두세요. 아래 셀은 파일이 제대로 있는지만 확인합니다.

In [ ]:
edf = glob.glob(f'{BIDS_ROOT}/sub-*/**/*_eeg.edf', recursive=True)
print('EDF 파일 수:', len(edf))
print('EO:', sum('task-eyesOpen' in f for f in edf), '| EC:', sum('task-eyesClosed' in f for f in edf))

## 3. CBraMod 원본 코드 + 사전학습 가중치
`wjq-learning/CBraMod` 원본 코드와 `weighting666/CBraMod`의 `pretrained_weights.pth`를 Drive에 받아 둡니다. PD-EEG·AD-EEG 노트북도 이 폴더를 다시 씁니다.

In [ ]:
CBRAMOD_REPO, WEIGHTS = f'{WORK}/CBraMod', f'{WORK}/pretrained_weights.pth'
if not os.path.exists(f'{CBRAMOD_REPO}/models/cbramod.py'):
    !git clone -q https://github.com/wjq-learning/CBraMod "$CBRAMOD_REPO"
if not os.path.exists(WEIGHTS):
    from huggingface_hub import hf_hub_download
    shutil.copy(hf_hub_download('weighting666/CBraMod', 'pretrained_weights.pth'), WEIGHTS)
print('repo:', CBRAMOD_REPO, '| weights MB:', round(os.path.getsize(WEIGHTS) / 1e6, 1))   # 약 19.8 MB

## 4. 스크립트 파일 만들기

In [ ]:
%%writefile /content/mumtaz_cbramod_mdd.py
"""
Mumtaz (OpenNeuro/NEMAR nm000114) MDD vs HC: frozen CBraMod + 로지스틱 회귀
TDBRAIN 보고서(2026-11-02)의 프로토콜을 그대로 옮긴 버전.

선정 규칙
  - 라벨: 피험자 ID 접두어 (HS -> 0, MDDS -> 1)
  - 측정 조건: eyesOpen(EO), eyesClosed(EC)만 사용, P300 제외
  - 같은 조건의 기록이 여러 개면 첫 번째(run-1 / 첫 세션)만 사용 (HS15 EO)
  - EO/EC가 하나도 없는 피험자 제외 (MDDS8)
  -> 63명 (HC 30 / MDD 33), 기록 119개 예상

전처리: 19채널(Mumtaz 순서) -> 0.3-75 Hz 대역통과 -> 50 Hz 노치 -> 200 Hz 리샘플링
        -> V->uV -> 5초 비중첩 구간(잔여 제외) -> 100 uV 초과 표시 -> /100 -> [n, 19, 5, 200]
특징: frozen CBraMod, proj_out=Identity, 시간 패치 5개 평균 -> 19x200 = 3,800차원
분류: StandardScaler + L2 로지스틱 회귀(C=0.01), 구간 단위 학습, 임계값 0.5
평가: 피험자 단위 층화 5-fold(shuffle, seed 42), 피험자 점수 = EO 평균과 EC 평균의 동일 가중 평균

사용법 (Colab)
  python mumtaz_cbramod_mdd.py extract --bids-root /content/nm000114 --out OUT \
      --cbramod-repo /content/CBraMod --weights pretrained_weights.pth
  python mumtaz_cbramod_mdd.py cv --out OUT --cond both      # EO+EC (주 결과)
  python mumtaz_cbramod_mdd.py cv --out OUT --cond EO
  python mumtaz_cbramod_mdd.py cv --out OUT --cond EC
  python mumtaz_cbramod_mdd.py cv --out OUT --cond both --max-sec 120   # TDBRAIN과 길이 맞춤(보조)
"""
import argparse
import glob
import json
import os
import re
import sys

import numpy as np
import pandas as pd

CHANNELS = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz",
            "C4", "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2"]
ALIASES = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}  # 혹시 10-10 이름으로 바뀐 경우 대비
SFREQ = 200
SEG_SEC = 5
SEG_LEN = SFREQ * SEG_SEC
SEED = 42


# ---------------------------------------------------------------- 선정
def select_recordings(bids_root):
    rows = []
    files = sorted(glob.glob(os.path.join(bids_root, "sub-*", "**", "*_eeg.edf"), recursive=True))
    for f in files:
        name = os.path.basename(f)
        if "task-eyesOpen" in name:
            cond = "EO"
        elif "task-eyesClosed" in name:
            cond = "EC"
        else:
            continue  # P300 제외
        sub = re.search(r"sub-([A-Za-z]+\d+)", f).group(1)
        if sub.startswith("HS"):
            label = 0
        elif sub.startswith("MDDS"):
            label = 1
        else:
            continue
        ses = re.search(r"ses-([^/_]+)", f)
        run = re.search(r"run-(\d+)", name)
        rows.append(dict(sub=sub, label=label, cond=cond, path=f,
                         ses=ses.group(1) if ses else "",
                         run=int(run.group(1)) if run else 0))
    df = pd.DataFrame(rows)
    # 같은 피험자·조건의 기록이 여러 개면 첫 세션, 첫 run만 사용
    df = (df.sort_values(["sub", "cond", "ses", "run", "path"])
            .groupby(["sub", "cond"], as_index=False).first())
    return df.reset_index(drop=True)


# ---------------------------------------------------------------- 전처리
def _norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    c = re.split(r"[-\s]", c)[0]
    return c


def load_and_segment(path):
    import mne
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    lookup = {}
    for ch in raw.ch_names:
        n = _norm(ch)
        n = {k.upper(): v for k, v in ALIASES.items()}.get(n, n)
        lookup.setdefault(n, ch)
    missing = [c for c in CHANNELS if c.upper() not in lookup]
    if missing:
        raise ValueError(f"{path}: 채널 없음 {missing} / 원본 채널 {raw.ch_names}")
    picks = [lookup[c.upper()] for c in CHANNELS]
    raw.pick(picks)
    raw.reorder_channels(picks)
    orig_sfreq = raw.info["sfreq"]
    orig_sec = raw.times[-1] + 1 / orig_sfreq

    raw.filter(0.3, 75.0, verbose=False)
    raw.notch_filter(50.0, verbose=False)
    raw.resample(SFREQ, verbose=False)

    x = raw.get_data() * 1e6  # V -> uV
    n = x.shape[1] // SEG_LEN  # 5초 미만 잔여 제외
    x = x[:, : n * SEG_LEN]
    segs = x.reshape(19, n, 5, SFREQ).transpose(1, 0, 2, 3)  # [n, 19, 5, 200]
    flag = (np.abs(segs) > 100).reshape(n, -1).any(axis=1)   # 품질 표시(제외에는 미사용)
    return (segs / 100.0).astype(np.float32), flag, orig_sfreq, orig_sec


# ---------------------------------------------------------------- CBraMod
def load_cbramod(repo, weights, device):
    import torch
    import torch.nn as nn
    sys.path.insert(0, repo)
    from models.cbramod import CBraMod
    model = CBraMod(in_dim=200, out_dim=200, d_model=200, dim_feedforward=800,
                    seq_len=30, n_layer=12, nhead=8)
    state = torch.load(weights, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    state = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(state)
    model.proj_out = nn.Identity()
    return model.to(device).eval()


def cmd_extract(a):
    import torch
    os.makedirs(a.out, exist_ok=True)
    recs = select_recordings(a.bids_root)
    n_sub = recs["sub"].nunique()
    print(f"선정: 피험자 {n_sub}명 (HC {recs[recs.label==0]['sub'].nunique()} / "
          f"MDD {recs[recs.label==1]['sub'].nunique()}), 기록 {len(recs)}개")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = load_cbramod(a.cbramod_repo, a.weights, device)

    feats, meta, log = [], [], []
    for i, r in recs.iterrows():
        segs, flag, sf, sec = load_and_segment(r.path)
        out = []
        with torch.no_grad():
            for b in range(0, len(segs), a.batch):
                xb = torch.from_numpy(segs[b:b + a.batch]).to(device)
                yb = model(xb)                      # [B, 19, 5, 200]
                out.append(yb.mean(dim=2).reshape(len(xb), -1).cpu().numpy())  # 패치 평균 -> 3,800
        feats.append(np.concatenate(out).astype(np.float32))
        for k in range(len(segs)):
            meta.append(dict(sub=r["sub"], label=r.label, cond=r.cond, seg=k, over100=bool(flag[k])))
        log.append(dict(sub=r["sub"], label=r.label, cond=r.cond, file=os.path.basename(r.path),
                        orig_sfreq=sf, orig_sec=round(sec, 2), n_seg=len(segs),
                        over100_ratio=round(float(flag.mean()), 4)))
        print(f"[{i+1}/{len(recs)}] {r['sub']} {r.cond}: {len(segs)}구간, 100uV 초과 {flag.mean():.2%}")

    np.save(os.path.join(a.out, "features.npy"), np.concatenate(feats))
    pd.DataFrame(meta).to_csv(os.path.join(a.out, "segments.csv"), index=False)
    pd.DataFrame(log).to_csv(os.path.join(a.out, "extract_log.csv"), index=False)
    print("저장 완료:", a.out)


# ---------------------------------------------------------------- 교차검증
def _metrics(y, p):
    from sklearn.metrics import (roc_auc_score, average_precision_score, balanced_accuracy_score,
                                 accuracy_score, f1_score, confusion_matrix)
    yhat = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return dict(AUROC=roc_auc_score(y, p), AUPRC=average_precision_score(y, p),
                BalAcc=balanced_accuracy_score(y, yhat), Acc=accuracy_score(y, yhat),
                Sens=tp / (tp + fn), Spec=tn / (tn + fp), F1=f1_score(y, yhat),
                TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))


def cmd_cv(a):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import StandardScaler

    X = np.load(os.path.join(a.out, "features.npy"))
    seg = pd.read_csv(os.path.join(a.out, "segments.csv"))
    keep = np.ones(len(seg), bool)
    if a.cond != "both":
        keep &= (seg.cond == a.cond).values
    if a.max_sec:
        keep &= (seg.seg < a.max_sec // SEG_SEC).values
    X, seg = X[keep], seg[keep].reset_index(drop=True)

    subs = seg.groupby("sub")["label"].first()
    sub_ids, y_sub = subs.index.values, subs.values
    print(f"조건 {a.cond}{' / 앞 ' + str(a.max_sec) + '초' if a.max_sec else ''}: "
          f"피험자 {len(sub_ids)}명 (HC {(y_sub==0).sum()} / MDD {(y_sub==1).sum()}), "
          f"기록 {seg.groupby(['sub','cond']).ngroups}개, 구간 {len(seg)}개")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    seg_p = np.zeros(len(seg))
    sub_rows, fold_rows = [], []
    for fold, (tr, te) in enumerate(skf.split(sub_ids, y_sub)):
        tr_m = seg["sub"].isin(sub_ids[tr]).values
        te_m = seg["sub"].isin(sub_ids[te]).values
        sc = StandardScaler().fit(X[tr_m])
        clf = LogisticRegression(C=0.01, max_iter=5000).fit(sc.transform(X[tr_m]), seg.label[tr_m])
        seg_p[te_m] = clf.predict_proba(sc.transform(X[te_m]))[:, 1]

        te_seg = seg[te_m].assign(p=seg_p[te_m])
        # 조건별 구간 평균 -> 조건들을 같은 비중으로 평균 (한 조건만 있으면 그 값)
        sp = te_seg.groupby(["sub", "cond"])["p"].mean().groupby("sub").mean()
        for s, p in sp.items():
            sub_rows.append(dict(sub=s, label=int(subs[s]), p=p, fold=fold))
        fy = np.array([subs[s] for s in sp.index])
        fold_rows.append(dict(fold=fold, **_metrics(fy, sp.values)))

    oof = pd.DataFrame(sub_rows).sort_values("sub")
    res = _metrics(oof.label.values, oof.p.values)
    fd = pd.DataFrame(fold_rows)
    res["fold_AUROC"] = f"{fd.AUROC.mean():.4f} ± {fd.AUROC.std(ddof=1):.4f}"
    res["fold_BalAcc"] = f"{fd.BalAcc.mean()*100:.2f} ± {fd.BalAcc.std(ddof=1)*100:.2f}%p"
    sm = _metrics(seg.label.values, seg_p)
    res["seg_AUROC"], res["seg_BalAcc"] = sm["AUROC"], sm["BalAcc"]
    res["n_sub"], res["n_seg"] = int(len(sub_ids)), int(len(seg))

    tag = f"cv_{a.cond}" + (f"_{a.max_sec}s" if a.max_sec else "")
    od = os.path.join(a.out, tag)
    os.makedirs(od, exist_ok=True)
    oof.to_csv(os.path.join(od, "oof_subject.csv"), index=False)
    fd.to_csv(os.path.join(od, "fold_metrics.csv"), index=False)
    with open(os.path.join(od, "metrics.json"), "w") as f:
        json.dump({k: (float(v) if isinstance(v, (np.floating,)) else v) for k, v in res.items()},
                  f, ensure_ascii=False, indent=2)

    for k in ["AUROC", "BalAcc", "Acc", "Sens", "Spec", "F1", "AUPRC"]:
        print(f"  {k:7s} {res[k]:.4f}")
    print(f"  혼동행렬 TN/FP/FN/TP = {res['TN']}/{res['FP']}/{res['FN']}/{res['TP']}")
    print(f"  fold AUROC {res['fold_AUROC']} | fold BalAcc {res['fold_BalAcc']}")
    print(f"  구간 단위 AUROC {res['seg_AUROC']:.4f} / BalAcc {res['seg_BalAcc']:.4f}")
    print("저장:", od)


def main():
    ap = argparse.ArgumentParser()
    sp = ap.add_subparsers(dest="cmd", required=True)
    e = sp.add_parser("extract")
    e.add_argument("--bids-root", required=True)
    e.add_argument("--out", required=True)
    e.add_argument("--cbramod-repo", required=True)
    e.add_argument("--weights", required=True)
    e.add_argument("--batch", type=int, default=64)
    c = sp.add_parser("cv")
    c.add_argument("--out", required=True)
    c.add_argument("--cond", choices=["both", "EO", "EC"], default="both")
    c.add_argument("--max-sec", type=int, default=0)
    a = ap.parse_args()
    cmd_extract(a) if a.cmd == "extract" else cmd_cv(a)


if __name__ == "__main__":
    main()

## 5. 특징 추출
선정 결과가 **피험자 63명 (HC 30 / MDD 33), 기록 119개**로 나오면 정상

In [ ]:
!python /content/mumtaz_cbramod_mdd.py extract --bids-root "$BIDS_ROOT" --out "$OUT" --cbramod-repo "$CBRAMOD_REPO" --weights "$WEIGHTS"

In [ ]:
import pandas as pd
log = pd.read_csv(f'{OUT}/extract_log.csv')
print(log.groupby(['label', 'cond'])[['n_seg', 'over100_ratio', 'orig_sec']].agg(['count', 'mean']).round(3))

## 6. 5-fold 평가: EO + EC (주 결과)

In [ ]:
!python /content/mumtaz_cbramod_mdd.py cv --out "$OUT" --cond both

## 7. 조건별 단독 평가: EO만 / EC만

In [ ]:
!python /content/mumtaz_cbramod_mdd.py cv --out "$OUT" --cond EO
!python /content/mumtaz_cbramod_mdd.py cv --out "$OUT" --cond EC

## 8. 보조: 앞 120초만 (TDBRAIN과 기록 길이 맞춤)

In [ ]:
!python /content/mumtaz_cbramod_mdd.py cv --out "$OUT" --cond both --max-sec 120

## 9. 보고서용 요약

In [ ]:
import json
rows = []
for tag in ['cv_both', 'cv_EO', 'cv_EC', 'cv_both_120s']:
    p = f'{OUT}/{tag}/metrics.json'
    if os.path.exists(p):
        m = json.load(open(p))
        rows.append({'setting': tag, 'n_sub': m['n_sub'], 'n_seg': m['n_seg'],
                     **{k: round(m[k], 4) for k in ['AUROC', 'BalAcc', 'Sens', 'Spec', 'AUPRC']},
                     'fold AUROC': m['fold_AUROC']})
pd.DataFrame(rows)